In [23]:
import pandas as pd
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
# Baseline model name (or full path). Prefix with 'baseline:' to pull from
# results/whisper_baseline/whisper_baseline.csv
BASELINE = 'baseline:whisper'

# Comparison model names (without .csv) or full paths — one or more.
COMPARISON_MODELS = [
    'bridge_dtw_eps_0.5',
    'bridge_dtw_eps_0.5_odesampling',
    'bridge_position_eps_1.5',
    'bridge_dtw_fixed_eps_0.5',
    'bridge_dtw_fixed_eps_0.5_odesampling',
    # 'bridge_dtw_fixed_eps_0.5_antirep',
    'bridge_dtw_fixed_x0_0.5',
    # 'bridge_dtw_fixed_x0_0.5_antirep',
    'bridge_dtw_fixed_x0_0.5_odesampling',
]

# Metric that drives win/draw/loss classification and best/worst ranking.
PRIMARY_METRIC = 'utt_wer'

# Columns to show in comparison tables. Use None to show all shared metric columns.
SHOW_COLS = ['utt_wer']

# Number of rows to show in the best-wins / worst-losses tables.
TOP_N = 20

# Optional filters (set to None to skip) — applied to baseline and all comparison models
FILTER_L1 = None       # e.g. 'Hindi'
FILTER_SPEAKER = None  # e.g. 'THV'

# ── Path resolution ───────────────────────────────────────────────────────────
ROOT = Path("/vol/gpudata/tsv22-fyp/accent-robust-asr")
BRIDGE_DIR   = Path(f'{ROOT}/results/bridge_eval')
STEERING_DIR = Path(f'{ROOT}/results/e2_steering')
BASELINE_DIR = Path(f'{ROOT}/results/whisper_baseline')

def resolve(name: str) -> Path:
    if name.startswith('baseline:'):
        return BASELINE_DIR / 'whisper_baseline.csv'
    p = Path(name)
    if p.suffix == '.csv' and p.exists():
        return p
    for d in [BRIDGE_DIR, STEERING_DIR]:
        candidate = d / f'{name}.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Cannot find CSV for {name!r}')

path_baseline = resolve(BASELINE)
print(f'Baseline: {BASELINE} -> {path_baseline}')
for name in COMPARISON_MODELS:
    print(f'  {name} -> {resolve(name)}')

Baseline: baseline:whisper -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/whisper_baseline/whisper_baseline.csv
  bridge_dtw_eps_0.5 -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_eps_0.5.csv
  bridge_dtw_eps_0.5_odesampling -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_eps_0.5_odesampling.csv
  bridge_position_eps_1.5 -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_position_eps_1.5.csv
  bridge_dtw_fixed_eps_0.5 -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_fixed_eps_0.5.csv
  bridge_dtw_fixed_eps_0.5_odesampling -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_fixed_eps_0.5_odesampling.csv
  bridge_dtw_fixed_x0_0.5 -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_fixed_x0_0.5.csv
  bridge_dtw_fixed_x0_0.5_odesampling -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_fixed_x0_0.5_odesampling.csv


In [24]:
NON_METRIC = {'utterance_id', 'speaker', 'l1', 'domain', 'wav_path', 'text',
              'prediction', 'reference_norm', 'prediction_norm', 'speaker_type',
              'bridge_split', '_label'}

def load(path: Path, label: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # normalise column names across different eval scripts
    renames = {'wer': 'utt_wer', 'mer': 'utt_mer', 'per': 'utt_per',
               'whisper_pred': 'prediction', 'whisper_pred_norm': 'prediction_norm'}
    df = df.rename(columns={k: v for k, v in renames.items() if k in df.columns})
    df['_label'] = label
    return df

def apply_filters(df):
    if FILTER_L1 and 'l1' in df.columns:
        df = df[df['l1'] == FILTER_L1]
    if FILTER_SPEAKER and 'speaker' in df.columns:
        df = df[df['speaker'] == FILTER_SPEAKER]
    return df

def compare(da: pd.DataFrame, model_name: str) -> dict:
    """Join filtered baseline rows `da` against `model_name`'s eval CSV and
    precompute win/draw/loss stats, an L1 breakdown, and best/worst tables.

    Columns from the baseline get a `_baseline` suffix, columns from the
    comparison model get a `_bridge` suffix. `{metric}_delta` is always
    `bridge - baseline`: positive means the comparison model is worse than
    the baseline (higher WER), negative means it's better (lower WER).
    """
    db = apply_filters(load(resolve(model_name), model_name))

    metric_cols_a = [c for c in da.columns if c not in NON_METRIC]
    metric_cols_b = [c for c in db.columns if c not in NON_METRIC]
    shared_metrics = [c for c in metric_cols_a if c in metric_cols_b]
    cols_to_show = SHOW_COLS if SHOW_COLS else shared_metrics

    keep_a = ['utterance_id', 'speaker'] + (['l1'] if 'l1' in da.columns else []) + \
             ['text'] + cols_to_show + \
             (['prediction_norm'] if 'prediction_norm' in da.columns else [])
    keep_b = ['utterance_id', 'speaker'] + cols_to_show + \
             (['prediction_norm'] if 'prediction_norm' in db.columns else [])

    merged = da[keep_a].merge(
        db[[c for c in keep_b if c in db.columns]],
        on=['utterance_id', 'speaker'],
        suffixes=('_baseline', '_bridge'),
        how='inner'
    )

    for c in cols_to_show:
        c_base, c_bridge = f'{c}_baseline', f'{c}_bridge'
        if c_base in merged.columns and c_bridge in merged.columns:
            merged[f'{c}_delta'] = merged[c_bridge] - merged[c_base]

    pm_delta = f'{PRIMARY_METRIC}_delta'
    pm_bridge = f'{PRIMARY_METRIC}_bridge'
    is_win  = merged[pm_delta] < 0
    is_loss = merged[pm_delta] > 0
    is_draw = merged[pm_delta] == 0
    is_zero = merged[pm_bridge] == 0
    winloss_counts = {
        'wins':          int(is_win.sum()),
        'corrected':     int((is_win & is_zero).sum()),    # win that lands on 0 WER
        'improved':      int((is_win & ~is_zero).sum()),   # win that's still nonzero
        'draws':         int(is_draw.sum()),
        'draws_zero':    int((is_draw & is_zero).sum()),   # tied at 0 WER
        'draws_nonzero': int((is_draw & ~is_zero).sum()),  # tied but nonzero
        'losses':        int(is_loss.sum()),
    }

    l1_table = None
    if 'l1' in merged.columns:
        rows = []
        for l1, g in merged.groupby('l1'):
            row = {'l1': l1, 'n': len(g)}
            for c in cols_to_show:
                c_base, c_bridge = f'{c}_baseline', f'{c}_bridge'
                row[f'{c}_baseline'] = g[c_base].mean()
                row[f'{c}_bridge'] = g[c_bridge].mean()
                row[f'{c}_delta'] = g[c_bridge].mean() - g[c_base].mean()
            rows.append(row)
        l1_table = pd.DataFrame(rows).set_index('l1').round(4)

    text_cols   = ['utterance_id', 'speaker'] + (['l1'] if 'l1' in merged.columns else []) + ['text']
    pred_cols   = [c for c in merged.columns if 'prediction_norm' in c]
    metric_disp = [c for c in merged.columns if any(c.startswith(m) for m in cols_to_show)]
    browse_cols = text_cols + pred_cols + metric_disp

    best_wins    = merged.sort_values(pm_delta, ascending=True).head(TOP_N)[browse_cols].reset_index(drop=True)
    worst_losses = merged.sort_values(pm_delta, ascending=False).head(TOP_N)[browse_cols].reset_index(drop=True)

    return {
        'name': model_name,
        'merged': merged,
        'n': len(merged),
        'cols_to_show': cols_to_show,
        'winloss_counts': winloss_counts,
        'l1_table': l1_table,
        'best_wins': best_wins,
        'worst_losses': worst_losses,
    }

da_baseline = apply_filters(load(path_baseline, BASELINE))
print(f'Baseline {BASELINE!r}: {len(da_baseline)} rows')

results = [compare(da_baseline, name) for name in COMPARISON_MODELS]

print(f'\nComputed {len(results)} comparison(s) against baseline:')
for r in results:
    wl = r['winloss_counts']
    print(f"  {r['name']:35s} n={r['n']:5d}  wins={wl['wins']:4d} (corrected={wl['corrected']:4d}, improved={wl['improved']:4d})"
          f"  draws={wl['draws']:4d} (zero={wl['draws_zero']:4d}, nonzero={wl['draws_nonzero']:4d})  losses={wl['losses']:4d}")

Baseline 'baseline:whisper': 31395 rows

Computed 7 comparison(s) against baseline:
  bridge_dtw_eps_0.5                  n= 7796  wins= 635 (corrected= 180, improved= 455)  draws=5935 (zero=3156, nonzero=2779)  losses=1225
  bridge_dtw_eps_0.5_odesampling      n= 7796  wins= 200 (corrected=  45, improved= 155)  draws=7096 (zero=3396, nonzero=3700)  losses= 499
  bridge_position_eps_1.5             n= 7796  wins=  44 (corrected=   4, improved=  40)  draws= 118 (zero=  27, nonzero=  91)  losses=7633
  bridge_dtw_fixed_eps_0.5            n= 7796  wins= 981 (corrected= 312, improved= 669)  draws=5811 (zero=3195, nonzero=2616)  losses=1003
  bridge_dtw_fixed_eps_0.5_odesampling n= 7796  wins=   5 (corrected=   1, improved=   4)  draws=7780 (zero=3484, nonzero=4296)  losses=  10
  bridge_dtw_fixed_x0_0.5             n= 7796  wins= 982 (corrected= 328, improved= 654)  draws=5423 (zero=3106, nonzero=2317)  losses=1390
  bridge_dtw_fixed_x0_0.5_odesampling n= 7796  wins=1038 (corrected= 333, i

In [25]:
from IPython.display import display, Markdown

pm = PRIMARY_METRIC
raw_rows, pct_rows = [], []
for r in results:
    merged, wl, n = r['merged'], r['winloss_counts'], r['n']
    baseline_mean = merged[f'{pm}_baseline'].mean()
    bridge_mean   = merged[f'{pm}_bridge'].mean()

    raw_rows.append({
        'model': r['name'],
        'n': n,
        f'{pm}_baseline': baseline_mean,
        f'{pm}_bridge': bridge_mean,
        f'{pm}_delta': bridge_mean - baseline_mean,
        'wins': wl['wins'],
        'corrected': wl['corrected'],
        'improved': wl['improved'],
        'draws': wl['draws'],
        'draws_zero': wl['draws_zero'],
        'draws_nonzero': wl['draws_nonzero'],
        'losses': wl['losses'],
    })
    pct_rows.append({
        'model': r['name'],
        'win_%':           wl['wins']          / n * 100,
        'corrected_%':     wl['corrected']     / n * 100,
        'improved_%':      wl['improved']      / n * 100,
        'draw_%':          wl['draws']         / n * 100,
        'draws_zero_%':    wl['draws_zero']    / n * 100,
        'draws_nonzero_%': wl['draws_nonzero'] / n * 100,
        'loss_%':          wl['losses']        / n * 100,
    })

overview_raw = pd.DataFrame(raw_rows).set_index('model').round(4)
overview_pct = pd.DataFrame(pct_rows).set_index('model').round(4)

display(Markdown('**Raw counts (wins split into corrected [→0 WER] / improved [still nonzero]; '
                 'draws split into zero / nonzero):**'))
display(overview_raw)
display(Markdown('**Percentages (of compared utterances `n`):**'))
display(overview_pct)

**Raw counts (wins split into corrected [→0 WER] / improved [still nonzero]; draws split into zero / nonzero):**

,n,utt_wer_baseline,utt_wer_bridge,utt_wer_delta,wins,corrected,improved,draws,draws_zero,draws_nonzero,losses
model,,,,,,,,,,,
bridge_dtw_eps_0.5,7796,0.154,0.1915,0.0374,635,180,455,5935,3156,2779,1225
bridge_dtw_eps_0.5_odesampling,7796,0.154,0.2285,0.0745,200,45,155,7096,3396,3700,499
bridge_position_eps_1.5,7796,0.154,6.7746,6.6206,44,4,40,118,27,91,7633
bridge_dtw_fixed_eps_0.5,7796,0.154,0.1941,0.0400,981,312,669,5811,3195,2616,1003
bridge_dtw_fixed_eps_0.5_odesampling,7796,0.154,0.1541,0.0001,5,1,4,7780,3484,4296,10
bridge_dtw_fixed_x0_0.5,7796,0.154,0.4194,0.2653,982,328,654,5423,3106,2317,1390
bridge_dtw_fixed_x0_0.5_odesampling,7796,0.154,0.3005,0.1465,1038,333,705,5667,3195,2472,1090


**Percentages (of compared utterances `n`):**

,win_%,corrected_%,improved_%,draw_%,draws_zero_%,draws_nonzero_%,loss_%
model,,,,,,,
bridge_dtw_eps_0.5,8.1452,2.3089,5.8363,76.1288,40.4823,35.6465,15.7132
bridge_dtw_eps_0.5_odesampling,2.5654,0.5772,1.9882,91.0210,43.5608,47.4602,6.4007
bridge_position_eps_1.5,0.5644,0.0513,0.5131,1.5136,0.3463,1.1673,97.9092
bridge_dtw_fixed_eps_0.5,12.5834,4.0021,8.5813,74.5382,40.9826,33.5557,12.8656
bridge_dtw_fixed_eps_0.5_odesampling,0.0641,0.0128,0.0513,99.7948,44.6896,55.1052,0.1283
bridge_dtw_fixed_x0_0.5,12.5962,4.2073,8.3889,69.5613,39.8409,29.7204,17.8297
bridge_dtw_fixed_x0_0.5_odesampling,13.3145,4.2714,9.0431,72.6911,40.9826,31.7086,13.9815


In [26]:
from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 200)

for r in results:
    display(Markdown(f"## {r['name']}"))

    wl, n = r['winloss_counts'], r['n']
    print(f"Utterances compared: {n}")
    print(f"  wins   (model better than baseline): {wl['wins']:5d}  ({wl['wins']/n*100:5.2f}%)")
    print(f"  draws  (tied):                       {wl['draws']:5d}  ({wl['draws']/n*100:5.2f}%)")
    print(f"  losses (model worse than baseline):  {wl['losses']:5d}  ({wl['losses']/n*100:5.2f}%)")

    if r['l1_table'] is not None:
        display(Markdown("**By L1:**"))
        display(r['l1_table'])

    display(Markdown(f"**Best wins — top {TOP_N} by `{PRIMARY_METRIC}` delta (model beats baseline most):**"))
    display(r['best_wins'])

    display(Markdown(f"**Worst losses — top {TOP_N} by `{PRIMARY_METRIC}` delta (model loses to baseline most):**"))
    display(r['worst_losses'])

## bridge_dtw_eps_0.5

Utterances compared: 7796
  wins   (model better than baseline):   635  ( 8.15%)
  draws  (tied):                        5935  (76.13%)
  losses (model worse than baseline):   1225  (15.71%)


**By L1:**

,n,utt_wer_baseline,utt_wer_bridge,utt_wer_delta
l1,,,,
Arabic,1132,0.1308,0.1399,0.0091
Chinese,1130,0.2010,0.2965,0.0955
English,1132,0.0402,0.0448,0.0046
Hindi,1132,0.0708,0.0738,0.0030
Korean,1131,0.0855,0.0902,0.0047
Spanish,1007,0.2364,0.2838,0.0474
Vietnamese,1132,0.3223,0.4214,0.0991


**Best wins — top 20 by `utt_wer` delta (model beats baseline most):**

...

,n,utt_wer_baseline,utt_wer_bridge,utt_wer_delta
l1,,,,
Arabic,1132,0.1308,0.1795,0.0487
Chinese,1130,0.2010,0.3103,0.1093
English,1132,0.0402,0.0378,-0.0024
Hindi,1132,0.0708,0.0655,-0.0053
Korean,1131,0.0855,0.0818,-0.0037
Spanish,1007,0.2364,0.6224,0.3860
Vietnamese,1132,0.3223,0.8412,0.5189


**Best wins — top 20 by `utt_wer` delta (model beats baseline most):**

,utterance_id,speaker,l1,text,prediction_norm_baseline,prediction_norm_bridge,utt_wer_baseline,utt_wer_bridge,utt_wer_delta
0,arctic_a0589,BWC,Chinese,I was sick once typhoid,i will seek once time for it,i was sick once typhoid,1.000000,0.000000,-1.000000
1,arctic_a0381,SVBI,Hindi,My name's Ferguson,my name is fargusson,my names ferguson,1.000000,0.000000,-1.000000
2,arctic_b0425,BDL,English,"There were orange-green, gold-green, and a copper-green.",there were orange green gold green and a copper green,there were orangegreen goldgreen and a coppergreen,0.857143,0.000000,-0.857143
3,arctic_b0207,ZHAA,Arabic,I'm as good as a man she urged,amasgut ezaman she urged,im as good as a man she urged,0.750000,0.000000,-0.750000
4,arctic_b0299,HQTV,Vietnamese,Miss Brodie's smile was slightly sarcastic,misbroadies my words like this sarcastic,miss brodys smile was slightly sarcastic,0.833333,0.166667,-0.666667
5,arctic_a0381,HJK,Korean,My name's Ferguson,my name is ferguson,my names ferguson,0.666667,0.000000,-0.666667
6,arctic_b0166,ZHAA,Arabic,Fast but endure,fast buttontoer,fast but endure,0.666667,0.000000,-0.666667
7,arctic_a0389,BWC,Chinese,Mab she said,mab shes sad,mab she said,0.666667,0.000000,-0.666667
8,arctic_b0218,EBVS,Spanish,The issue was not in doubt,the issue was not in the up to date,the issue was not in doubt,0.666667,0.000000,-0.666667
9,arctic_a0155,HQTV,Vietnamese,Won't you draw up gentlemen,one youve rarred and the other youve chandlemen,one youve rarred in chanderman,1.600000,1.000000,-0.600000


**Worst losses — top 20 by `utt_wer` delta (model loses to baseline most):**

,utterance_id,speaker,l1,text,prediction_norm_baseline,prediction_norm_bridge,utt_wer_baseline,utt_wer_bridge,utt_wer_delta
0,arctic_b0134,HQTV,Vietnamese,Blind with rage he darted in,life with earth he doubted in,life is a life that is not a life that is not a life that is not a life that...,0.500000,74.000000,73.500000
1,arctic_a0481,EBVS,Spanish,Nimrod replied with a slight manifestation of sensitiveness,nimrod replied with the lies manifestation of sensitiveness,nimrod replied with the light of the light of the light of the light of the ...,0.250000,54.875000,54.625000
2,arctic_b0437,HQTV,Vietnamese,Nowhere in the North is the soil so prolific,nowhere in the north either soso or from leific,no where in the north it is so so so so so so so so so so so so so so so so ...,0.555556,48.777778,48.222222
3,arctic_a0319,EBVS,Spanish,And the Edinburgh Evening News says with editorial gloom,and edimborn avenue news says with editorial groom,and the edinbauer news news news news news news news news news news news new...,0.444444,48.666667,48.222222
4,arctic_b0539,HQTV,Vietnamese,You were making them talk shop Ruth charged him,you are making them the talk shop good chance him,you were making them the talk shop you were making them the talk shop you we...,0.444444,42.555556,42.111111
5,arctic_a0263,HQTV,Vietnamese,Joan looked triumphantly at Sheldon who bowed,charm looked triumphantly as sheldon who bowed,sharn looked triumphantly as sharn looked triumphantly as sharn looked trium...,0.285714,42.000000,41.714286
6,arctic_a0503,BWC,Chinese,His beady black eyes saw bargains where other men saw bankruptcy,he beat the black eye saw blank knees where other men saw bankruptcy,he beaded black eyes of the black eye of the black eye of the black eye of t...,0.545455,40.090909,39.545455
7,arctic_a0407,BWC,Chinese,Mercedes screamed cried laughed and manifested the chaotic abandonment of hy...,macedas cried laughed and manifested the chaos and abandonment of hysteria,mcc cried laughed and manifested the chaos of the chaos of the chaos of the ...,0.363636,39.272727,38.909091
8,arctic_a0560,EBVS,Spanish,His mouth opened words shaped vainly on his lips,his mouth opened wore shaped vainly on his lips,his mouth opened and his mouth opened and his mouth opened and his mouth ope...,0.111111,39.000000,38.888889
9,arctic_a0202,EBVS,Spanish,She turned fearing that Jacques might see what was in her face,she turned it fairing that jax might see what was in her face,she turned and turned and turned and turned and turned and turned and turned...,0.250000,36.833333,36.583333


In [27]:
# ── Look up specific utterances across all models ─────────────────────────────
# Add (utterance_id, speaker) pairs here to see how the baseline and every
# comparison model handled them, side by side.
LOOKUP_UTTERANCES = [
    ('arctic_a0484', 'BDL'),
    ('arctic_b0319', 'HQTV'),
]

lookup_keys = pd.DataFrame(LOOKUP_UTTERANCES, columns=['utterance_id', 'speaker'])

base_cols = ['utterance_id', 'speaker'] + (['l1'] if 'l1' in da_baseline.columns else []) + \
            ['text', 'prediction_norm', PRIMARY_METRIC]
wide = lookup_keys.merge(da_baseline[base_cols], on=['utterance_id', 'speaker'], how='left')
wide = wide.rename(columns={
    'prediction_norm': f'prediction_norm_{BASELINE}',
    PRIMARY_METRIC: f'{PRIMARY_METRIC}_{BASELINE}',
})

for r in results:
    sub = r['merged'][['utterance_id', 'speaker', 'prediction_norm_bridge', f'{PRIMARY_METRIC}_bridge']].rename(columns={
        'prediction_norm_bridge': f'prediction_norm_{r["name"]}',
        f'{PRIMARY_METRIC}_bridge': f'{PRIMARY_METRIC}_{r["name"]}',
    })
    wide = wide.merge(sub, on=['utterance_id', 'speaker'], how='left')

wide

,utterance_id,speaker,l1,text,prediction_norm_baseline:whisper,utt_wer_baseline:whisper,prediction_norm_bridge_dtw_eps_0.5,utt_wer_bridge_dtw_eps_0.5,prediction_norm_bridge_dtw_eps_0.5_odesampling,utt_wer_bridge_dtw_eps_0.5_odesampling,prediction_norm_bridge_position_eps_1.5,utt_wer_bridge_position_eps_1.5,prediction_norm_bridge_dtw_fixed_eps_0.5,utt_wer_bridge_dtw_fixed_eps_0.5,prediction_norm_bridge_dtw_fixed_eps_0.5_odesampling,utt_wer_bridge_dtw_fixed_eps_0.5_odesampling,prediction_norm_bridge_dtw_fixed_x0_0.5,utt_wer_bridge_dtw_fixed_x0_0.5,prediction_norm_bridge_dtw_fixed_x0_0.5_odesampling,utt_wer_bridge_dtw_fixed_x0_0.5_odesampling
0,arctic_a0484,BDL,English,No-sir-ee.,no surrey,2.0,no sirree,2.0,no sir re,3.0,no siri,2.0,no surrey,2.0,no surrey,2.0,no sirree,2.0,no surrey,2.0
1,arctic_b0319,HQTV,Vietnamese,Daylight was tired profoundly tired,they lied to a tire a foully tire,1.6,they lied were tired profoundly tired,0.6,they lied to a tire a foully tire,1.6,they lie lie lie lie lie lie lie lie lie lie lie lie lie lie lie lie lie lie...,44.6,they lied were tired were fowlly tired,1.0,they lied to a tire a foully tire,1.6,they lied to the tyrant and they were tired,1.6,they lied were tied refowlied tied,1.2
